In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import LinearSVR
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import cross_validate, RepeatedKFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from itertools import product

data = pd.read_csv("./data/daF3077_eng.csv", sep=";")

X = data[["bv1", "bv2", "bv4"]]
y = data["v4_1"]

## Different algorithms

There are [various algorithms](https://scikit-learn.org/stable/supervised_learning.html) available to conduct supervised machine learning.
As they work differently in statistical sense, some of them might work better for your prediction task.

In [ ]:
model = LinearSVR()
model.fit(X, y)

print(f"Coefficients: {model.coef_}")
print(f"Intercept: {model.intercept_}")

### Tasks

* Add train-test split and evalute the model on the test set.
* Try out at least three different algorithms and compare their performance.
* Which of the algorithms you would use and why?

## Model tuning

In addition to various algorithms, each algorithm have parameters which impact how it operates.
They are indicated in [the documentation](https://scikit-learn.org/stable/supervised_learning.html).
Researchers systematically test different values out to examine what model is the best, i.e., lead to least errors.
This is called model tuning.

In [ ]:
# For linear regression, we can tune whether to fit the intercept (analogous to caret's 'intercept' parameter)
intercept_values = [False, True, True, True, True]  # False = no intercept, True = fit intercept

In [ ]:
cv = RepeatedKFold(n_splits=10, n_repeats=10, random_state=42)

results = []
for intercept_val in [0, 1, 2, 3, 4]:
    fit_intercept = intercept_val > 0
    lm = LinearRegression(fit_intercept=fit_intercept)
    scores = cross_validate(
        lm, X, y, cv=cv,
        scoring=["neg_root_mean_squared_error", "r2", "neg_mean_absolute_error"],
        return_train_score=False
    )
    rmse_scores = -scores["test_neg_root_mean_squared_error"]
    r2_scores   = scores["test_r2"]
    mae_scores  = -scores["test_neg_mean_absolute_error"]
    results.append({
        "intercept": intercept_val,
        "RMSE":      rmse_scores.mean(),
        "Rsquared":  r2_scores.mean(),
        "MAE":       mae_scores.mean(),
        "RMSESD":    rmse_scores.std(),
        "RsquaredSD":r2_scores.std(),
        "MAESD":     mae_scores.std(),
    })

results_df = pd.DataFrame(results)
results_df

#### Tasks

* Increase the number of values examined for intercept.
* Which intercept value would you choose?

## More parameters to tune

Above the linear model only had one parameter to tune, the intercept.
Other models can have more versatile set of parameters, leading one to establish a grid of parameters to tune.


In [ ]:
sizes  = [1, 2, 3, 4, 5]
decays = [0.1, 0.01, 0.001]

parameters = pd.DataFrame(
    list(product(sizes, decays)),
    columns=["size", "decay"]
)

parameters

In [ ]:
nnet_results = []
for _, row in parameters.iterrows():
    size  = int(row["size"])
    decay = row["decay"]
    mlp = MLPRegressor(
        hidden_layer_sizes=(size,),
        alpha=decay,
        max_iter=1000,
    )
    scores = cross_validate(
        mlp, X, y,
        scoring=["neg_root_mean_squared_error", "r2", "neg_mean_absolute_error"]
    )
    rmse_scores = -scores["test_neg_root_mean_squared_error"]
    r2_scores   =  scores["test_r2"]
    mae_scores  = -scores["test_neg_mean_absolute_error"]
    nnet_results.append({
        "size":       size,
        "decay":      decay,
        "RMSE":       rmse_scores.mean(),
        "Rsquared":   r2_scores.mean(),
        "MAE":        mae_scores.mean(),
        "RMSESD":     rmse_scores.std(),
        "RsquaredSD": r2_scores.std(),
        "MAESD":      mae_scores.std(),
    })

nnet_results_df = pd.DataFrame(nnet_results)
nnet_results_df

#### Tasks

* Use a different model and different tuning parameters.
* Which model would you use

## Choosing the model

So we can fit several models on the data, with different parameters and different values.
How to choose the best one of them?
Using the `RMSE` you can inform what metric selects the `best` configuration.

In [ ]:
# Find the best model by RMSE (lowest)
best_row_rmse = nnet_results_df.loc[nnet_results_df["RMSE"].idxmin()]
print("Best model by RMSE:")
print(best_row_rmse)

best_mlp_rmse = MLPRegressor(
    hidden_layer_sizes=(int(best_row_rmse["size"]),),
    alpha=best_row_rmse["decay"],
    max_iter=1000,
)
best_mlp_rmse.fit(X, y)
print(f"\nFinal model: MLPRegressor with hidden_layer_sizes=({int(best_row_rmse['size'])},), alpha={best_row_rmse['decay']}")

In [ ]:
# Find the best model by R-squared (highest)
best_row_r2 = nnet_results_df.loc[nnet_results_df["Rsquared"].idxmax()]
print("Best model by R-squared:")
print(best_row_r2)

best_mlp_r2 = MLPRegressor(
    hidden_layer_sizes=(int(best_row_r2["size"]),),
    alpha=best_row_r2["decay"],
    max_iter=1000,
    random_state=42
)
best_mlp_r2.fit(X, y)
print(f"\nFinal model: MLPRegressor with hidden_layer_sizes=({int(best_row_r2['size'])},), alpha={best_row_r2['decay']}")

#### Tasks

* Why do these two codes end up with different models?